In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Fungsi Circuit IBM

> **Note:** * Qiskit Functions adalah ciri eksperimental yang hanya tersedia untuk pengguna Pelan Premium, Pelan Flex, dan Pelan On-Prem (melalui API IBM Quantum Platform) IBM Quantum&reg;. Ia berada dalam status keluaran pratonton dan tertakluk kepada perubahan.

## Gambaran keseluruhan
Fungsi Circuit IBM&reg; mengambil [PUB abstrak](./primitive-input-output) sebagai input, dan mengembalikan nilai jangkaan yang dimitigasi sebagai output. Fungsi Circuit ini merangkumi saluran paip automatik dan tersuai untuk membolehkan penyelidik memberi tumpuan kepada penemuan algoritma dan aplikasi.
## Huraian
Setelah menghantar PUB anda, Circuit abstrak dan boleh jadi anda secara automatik ditranspil, dilaksanakan pada perkakasan, dan diproses selepas untuk mengembalikan nilai jangkaan yang dimitigasi. Untuk melakukan ini, ia menggabungkan alat-alat berikut:

- [Perkhidmatan Transpiler Qiskit](./qiskit-transpiler-service), termasuk pemilihan auto laluan transpilasi dipacu AI dan heuristik untuk menterjemahkan Circuit abstrak anda ke Circuit ISA yang dioptimumkan untuk perkakasan
- [Penindasan dan mitigasi ralat yang diperlukan untuk pengiraan berskala utiliti](./error-mitigation-and-suppression-techniques), termasuk twirling pengukuran dan gate, penyahgandingan dinamik, Twirled Readout Error eXtinction (TREX), Zero-Noise Extrapolation (ZNE), dan Probabilistic Error Amplification (PEA)
- [Qiskit Runtime Estimator](./get-started-with-primitives), untuk melaksanakan PUB ISA pada perkakasan dan mengembalikan nilai jangkaan yang dimitigasi

![Fungsi Circuit IBM](../docs/images/guides/ibm-circuit-function/ibm-circuit-function.svg)
## Mula
Sahkan jati diri menggunakan [kunci API](http://quantum.cloud.ibm.com/) anda dan pilih Qiskit Function seperti berikut. (Petikan ini mengandaikan anda telah [menyimpan akaun anda](/guides/functions#install-qiskit-functions-catalog-client) ke persekitaran tempatan anda.)

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

function = catalog.load("ibm/circuit-function")

## Contoh
 Untuk bermula, cuba contoh asas ini:

In [2]:
from qiskit.circuit.random import random_circuit
from qiskit_ibm_runtime import QiskitRuntimeService

# You can skip this step if you have a target backend, e.g.
# backend_name = "ibm_brisbane"
# You'll need to specify the credentials when initializing QiskitRuntimeService, if they were not previously saved.
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit = random_circuit(num_qubits=2, depth=2, seed=42)
observable = "Z" * circuit.num_qubits
pubs = [(circuit, observable)]

job = function.run(
    # Use `backend_name=backend_name` if you didn't initialize a backend object
    backend_name=backend.name,
    pubs=pubs,
)

Semak [status](/guides/functions#check-job-status) beban kerja Qiskit Function anda atau kembalikan [keputusan](/guides/functions#retrieve-results) seperti berikut:

In [3]:
print(job.status())
result = job.result()

QUEUED


The results have the same format as an [Estimator result](/docs/guides/primitive-input-output#estimator-output):

In [4]:
print(f"The result of the submitted job had {len(result)} PUB\n")
print(
    f"The associated PubResult of this job has the following DataBins:\n {result[0].data}\n"
)
print(f"And this DataBin has attributes: {result[0].data.keys()}")
print(
    f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
)

The result of the submitted job had 1 PUB

The associated PubResult of this job has the following DataBins:
 DataBin(evs=np.ndarray(<shape=(), dtype=float64>), stds=np.ndarray(<shape=(), dtype=float64>), ensemble_standard_error=np.ndarray(<shape=(), dtype=float64>))

And this DataBin has attributes: dict_keys(['evs', 'stds', 'ensemble_standard_error'])
The expectation values measured from this PUB are: 
1.02116704805492


Keputusan mempunyai format yang sama seperti [keputusan Estimator](/guides/primitive-input-output#estimator-output):

In [5]:
options = {"mitigation_level": 2}

job = function.run(backend_name=backend.name, pubs=pubs, options=options)

#### All available options

In addition to `mitigation_level`, the IBM Circuit function also provides a select number of advanced options that allow you to fine-tune the cost/accuracy trade-off. The following table shows all the available options:

| Option | Sub-option | Sub-sub-option | Description | Choices | Default |
|-|-|-|-|-|-|
| default_precision |  |  | The default precision to use for any PUB or `run()`<br/>call that does not specify one.<br/>Each input PUB can specify its own precision. If the `run()` method is given a precision, then that value is used for all PUBs in the `run()` call that do not specify their own.  | float > 0 | 0.015625 |
| max_execution_time |  |  | Maximum execution time in seconds, which is based<br/>on QPU usage (not wall clock time). QPU usage is the<br/>amount of time that the QPU is dedicated to processing your job. If a job exceeds this time limit, it is forcibly canceled. | Integer number of seconds in the range [1, 10800] |  |
| mitigation_level |  |  | How much error suppression and mitigation to apply. Refer to the [Mitigation level](#mitigation-level) section for more information about the methods used at each level. | 1 / 2 / 3 | 1 |
| optimization_level |  |  | How much optimization to perform on the circuits. [Higher levels](/docs/guides/set-optimization) generate more optimized circuits, at the expense of longer transpilation time. | 1 / 2 / 3 | 2 |
| dynamical_decoupling | enable |  | Whether to enable dynamical decoupling. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#dynamical-decoupling) for the explanation of the method.  | True/False | True |
|  | sequence_type |  | Which dynamical decoupling sequence to use.<br/>* `XX`: use the sequence `tau/2 - (+X) - tau - (+X) - tau/2`<br/>* `XpXm`: use the sequence `tau/2 - (+X) - tau - (-X) - tau/2`<br/>* `XY4`: use the sequence<br/>`tau/2 - (+X) - tau - (+Y) - tau (-X) - tau - (-Y) - tau/2` | 'XX'/'XpXm'/'XY4' | 'XX' |
| twirling | enable_gates |  | Whether to apply 2-qubit Clifford gate twirling. | True/False | False |
|  | enable_measure |  | Whether to enable twirling of measurements. | True/False | True |
| resilience | measure_mitigation |  | Whether to enable TREX measurement error mitigation method. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#twirled-readout-error-extinction-trex) for the explanation of the method.  | True/False | True |
|  | zne_mitigation |  | Whether to turn on Zero Noise Extrapolation error mitigation method. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne) for the explanation of the method.  | True/False | False |
|  | zne | amplifier | Which technique to use for amplifying noise. One of: <br/> - `gate_folding` (default) uses 2-qubit gate folding to amplify noise. If the noise factor requires amplifying only a subset of the gates, then these gates are chosen randomly.<br/><br/> - `gate_folding_front` uses 2-qubit gate folding to amplify noise. If the noise factor requires amplifying only a subset of the gates, then these gates are selected from the front of the topologically ordered DAG circuit.<br/><br/> - `gate_folding_back` uses 2-qubit gate folding to amplify noise. If the noise factor requires amplifying only a subset of the gates, then these gates are selected from the back of the topologically ordered DAG circuit.<br/><br/> - `pea` uses a technique called Probabilistic error amplification (PEA) to amplify noise. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#probabilistic-error-amplification-pea) for the explanation of the method.  | gate_folding / gate_folding_front / gate_folding_back / pea | gate_folding |
|  |  | noise_factors | Noise factors to use for noise amplification. | list of floats; each float >= 1 | (1, 1.5, 2) for PEA, and (1, 3, 5) otherwise. |
|  |  | extrapolator | Noise factors to evaluate the fit extrapolation models at. This option does not affect execution or model fitting in any way; it only determines the points at which the `extrapolator` objects are evaluated to be returned in the data fields called `evs_extrapolated` and `stds_extrapolated`. | one or more of `exponential`,`linear`, `double_exponential`,`polynomial_degree_(1 <= k <= 7)` | (`exponential`, `linear`) |
|  | pec_mitigation |  | Whether to turn on Probabilistic Error Cancellation error mitigation method. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec) for the explanation of the method.  | True/False | False |
|  | pec | max_overhead | The maximum circuit sampling overhead allowed, or `None` for no maximum. | None/ integer >1 | 100 |

In the following example, setting the mitigation level to 1 initially turns off ZNE mitigation, but setting `zne_mitigation` to `True` overrides the relevant setup from `mitigation_level`.

In [6]:
options = {"mitigation_level": 1, "resilience": {"zne_mitigation": True}}

## Input
Lihat jadual berikut untuk semua parameter input yang diterima oleh Fungsi Circuit IBM. Bahagian [Pilihan](#options) seterusnya memberikan butiran lebih lanjut tentang `options` yang tersedia.
| Nama      | Jenis                       | Huraian                                                                                                                                                                                                                         | Diperlukan | Lalai                                                                  | Contoh                                  |
|-----------|----------------------------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|----------|--------------------------------------------------------------------------|------------------------------------------|
| backend_name   | str                        | Nama Backend untuk membuat pertanyaan.                                                                                                                                                                                              | Ya      | N/A                                                                      | `ibm_fez`                                |
| pubs      | Iterable[EstimatorPubLike] | Iterable objek seperti PUB (primitive unified bloc) abstrak, seperti tupel `(circuit, observables)` atau `(circuit, observables, parameter_values)`. Lihat [Gambaran keseluruhan PUB](/guides/primitive-input-output#overview-of-pubs) untuk maklumat lanjut. Circuit boleh abstrak (bukan ISA). | Ya      | N/A                                                                      | (circuit, observables, parameter_values) |
| options   | dict                       | Pilihan input. Lihat bahagian **Pilihan** untuk butiran lanjut.                                                                                                                                                                                | Tidak       | Lihat bahagian **Pilihan** untuk butiran.                                                   | `{"optimization_level": 3}`                |
| instance  | str                        | Nama sumber awan bagi instans yang ingin digunakan dalam format tersebut.                                                                                                                                                                                        | Tidak       | Satu dipilih secara rawak jika akaun anda mempunyai akses ke pelbagai instans. | `CRN`                   |
### Pilihan
#### Struktur
Sama seperti primitif Qiskit Runtime, pilihan untuk Fungsi Circuit IBM boleh ditentukan sebagai kamus bersarang. Pilihan yang kerap digunakan, seperti ``optimization_level`` dan ``mitigation_level``, berada di peringkat pertama. Pilihan lain yang lebih lanjutan dikelompokkan ke dalam kategori yang berbeza, seperti ``resilience``.

#### Lalai
Jika anda tidak menentukan nilai untuk sesuatu pilihan, nilai lalai yang ditentukan oleh perkhidmatan digunakan.

#### Tahap mitigasi
Fungsi Circuit IBM juga menyokong `mitigation_level`. Tahap mitigasi menentukan berapa banyak penindasan dan mitigasi ralat yang digunakan untuk kerja. Tahap yang lebih tinggi menghasilkan keputusan yang lebih tepat, dengan mengorbankan masa pemprosesan yang lebih lama. Darjah pengurangan ralat bergantung pada kaedah yang digunakan. Tahap mitigasi mengabstrakkan pilihan kaedah mitigasi dan penindasan ralat terperinci untuk membolehkan pengguna mempertimbangkan pertukaran kos/ketepatan yang sesuai untuk aplikasi mereka. Jadual berikut menunjukkan kaedah yang sepadan bagi setiap tahap.

> **Note:** Walaupun namanya serupa, `mitigation_level` menggunakan teknik yang berbeza daripada yang digunakan oleh `resilience_level` Estimator.

Sama seperti ``resilience_level`` dalam Qiskit Runtime Estimator, ``mitigation_level`` menentukan set pilihan terpilih asas. Sebarang pilihan yang anda tentukan secara manual selain daripada tahap mitigasi diterapkan di atas set pilihan asas yang ditentukan oleh tahap mitigasi. Oleh itu, pada prinsipnya, anda boleh menetapkan tahap mitigasi ke 1, tetapi kemudian mematikan mitigasi pengukuran, walaupun ini tidak disyorkan.

| **Tahap Mitigasi** | **Teknik** |
|:-:|:-:|
| 1 [Lalai] | Penyahgandingan dinamik + twirling pengukuran + TREX  |
| 2 | Tahap 1 + twirling gate + ZNE melalui lipatan gate |
| 3 | Tahap 1 + twirling gate + ZNE melalui PEA |

Contoh berikut menunjukkan cara menetapkan tahap mitigasi:

In [7]:
print(f"The result of the submitted job had {len(result)} PUB")
print(
    f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
)
print(f"And the associated metadata is: \n{result[0].metadata}")

The result of the submitted job had 1 PUB
The expectation values measured from this PUB are: 
1.02116704805492
And the associated metadata is: 
{'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32}


#### Semua pilihan yang tersedia
Selain daripada `mitigation_level`, Fungsi Circuit IBM juga menyediakan beberapa pilihan lanjutan terpilih yang membolehkan anda menala pertukaran kos/ketepatan dengan lebih halus. Jadual berikut menunjukkan semua pilihan yang tersedia:

| Pilihan | Sub-pilihan | Sub-sub-pilihan | Huraian | Pilihan | Lalai |
|-|-|-|-|-|-|
| default_precision |  |  | Ketepatan lalai yang digunakan untuk sebarang PUB atau panggilan `run()`<br/>yang tidak menentukan satu.<br/>Setiap PUB input boleh menentukan ketepatannya sendiri. Jika kaedah `run()` diberikan ketepatan, nilai itu digunakan untuk semua PUB dalam panggilan `run()` yang tidak menentukan ketepatan mereka sendiri. | float > 0 | 0.015625 |
| max_execution_time |  |  | Masa pelaksanaan maksimum dalam saat, yang berdasarkan<br/>penggunaan QPU (bukan masa jam dinding). Penggunaan QPU ialah<br/>jumlah masa yang QPU didedikasikan untuk memproses kerja anda. Jika kerja melebihi had masa ini, ia dibatalkan secara paksa. | Nombor bulat saat dalam julat [1, 10800] |  |
| mitigation_level |  |  | Berapa banyak penindasan dan mitigasi ralat yang digunakan. Rujuk bahagian [Tahap mitigasi](#mitigation-level) untuk maklumat lanjut tentang kaedah yang digunakan pada setiap tahap. | 1 / 2 / 3 | 1 |
| optimization_level |  |  | Berapa banyak pengoptimuman yang dilakukan pada Circuit. [Tahap yang lebih tinggi](/guides/set-optimization) menghasilkan Circuit yang lebih dioptimumkan, dengan mengorbankan masa transpilasi yang lebih lama. | 1 / 2 / 3 | 2 |
| dynamical_decoupling | enable |  | Sama ada untuk mengaktifkan penyahgandingan dinamik. Rujuk [Teknik penindasan dan mitigasi ralat](/guides/error-mitigation-and-suppression-techniques#dynamical-decoupling) untuk penjelasan kaedah tersebut.  | True/False | True |
|  | sequence_type |  | Urutan penyahgandingan dinamik yang hendak digunakan.<br/>* `XX`: gunakan urutan `tau/2 - (+X) - tau - (+X) - tau/2`<br/>* `XpXm`: gunakan urutan `tau/2 - (+X) - tau - (-X) - tau/2`<br/>* `XY4`: gunakan urutan<br/>`tau/2 - (+X) - tau - (+Y) - tau (-X) - tau - (-Y) - tau/2` | 'XX'/'XpXm'/'XY4' | 'XX' |
| twirling | enable_gates |  | Sama ada untuk menggunakan twirling gate Clifford 2-Qubit. | True/False | False |
|  | enable_measure |  | Sama ada untuk mengaktifkan twirling pengukuran. | True/False | True |
| resilience | measure_mitigation |  | Sama ada untuk mengaktifkan kaedah mitigasi ralat pengukuran TREX. Rujuk [Teknik penindasan dan mitigasi ralat](/guides/error-mitigation-and-suppression-techniques#twirled-readout-error-extinction-trex) untuk penjelasan kaedah tersebut.  | True/False | True |
|  | zne_mitigation |  | Sama ada untuk mengaktifkan kaedah mitigasi ralat Zero Noise Extrapolation. Rujuk [Teknik penindasan dan mitigasi ralat](/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne) untuk penjelasan kaedah tersebut.  | True/False | False |
|  | zne | amplifier | Teknik yang hendak digunakan untuk mengamplifikasi hingar. Salah satu daripada: <br/> - `gate_folding` (lalai) menggunakan lipatan gate 2-Qubit untuk mengamplifikasi hingar. Jika faktor hingar memerlukan pengamplifikasian hanya sebahagian gate, gate tersebut dipilih secara rawak.<br/><br/> - `gate_folding_front` menggunakan lipatan gate 2-Qubit untuk mengamplifikasi hingar. Jika faktor hingar memerlukan pengamplifikasian hanya sebahagian gate, gate tersebut dipilih dari hadapan litar DAG yang dipesan topologinya.<br/><br/> - `gate_folding_back` menggunakan lipatan gate 2-Qubit untuk mengamplifikasi hingar. Jika faktor hingar memerlukan pengamplifikasian hanya sebahagian gate, gate tersebut dipilih dari belakang litar DAG yang dipesan topologinya.<br/><br/> - `pea` menggunakan teknik yang dipanggil Probabilistic error amplification (PEA) untuk mengamplifikasi hingar. Rujuk [Teknik penindasan dan mitigasi ralat](/guides/error-mitigation-and-suppression-techniques#probabilistic-error-amplification-pea) untuk penjelasan kaedah tersebut.  | gate_folding / gate_folding_front / gate_folding_back / pea | gate_folding |
|  |  | noise_factors | Faktor hingar yang hendak digunakan untuk pengamplifikasian hingar. | senarai nombor titik terapung; setiap nombor >= 1 | (1, 1.5, 2) untuk PEA, dan (1, 3, 5) untuk yang lain. |
|  |  | extrapolator | Faktor hingar untuk menilai model ekstrapolasi penyesuaian. Pilihan ini tidak mempengaruhi pelaksanaan atau penyesuaian model dengan apa-apa cara; ia hanya menentukan titik di mana objek `extrapolator` dinilai untuk dikembalikan dalam medan data yang dipanggil `evs_extrapolated` dan `stds_extrapolated`. | satu atau lebih daripada `exponential`,`linear`, `double_exponential`,`polynomial_degree_(1 <= k <= 7)` | (`exponential`, `linear`) |
|  | pec_mitigation |  | Sama ada untuk mengaktifkan kaedah mitigasi ralat Probabilistic Error Cancellation. Rujuk [Teknik penindasan dan mitigasi ralat](/guides/error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec) untuk penjelasan kaedah tersebut.  | True/False | False |
|  | pec | max_overhead | Overhed pensampelan Circuit maksimum yang dibenarkan, atau `None` untuk tiada maksimum. | None/ integer >1 | 100 |

Dalam contoh berikut, menetapkan tahap mitigasi ke 1 pada mulanya mematikan mitigasi ZNE, tetapi menetapkan `zne_mitigation` ke `True` menggantikan tetapan berkaitan dari `mitigation_level`.

In [8]:
job = function.run(
    backend_name="bad_backend_name", pubs=pubs, options=options
)

print(job.result())

QiskitServerlessException: "Traceback (most recent call last):\n  File \"/runner/runner.py\", line 10, in run\n    func = CircuitFunction(**arguments)\n           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"/runner/circuit_function/circuit_function.py\", line 87, in __init__\n    self._backend = self._service.backend(\n                    ^^^^^^^^^^^^^^^^^^^^^^\n  File \"/usr/local/lib/python3.11/site-packages/qiskit_ibm_runtime/qiskit_runtime_service.py\", line 754, in backend\n    backends = self.backends(name, instance=instance, use_fractional_gates=use_fractional_gates)\n               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"/usr/local/lib/python3.11/site-packages/qiskit_ibm_runtime/qiskit_runtime_service.py\", line 497, in backends\n    raise QiskitBackendNotFoundError(\"No backend matches the criteria.\")\nqiskit.providers.exceptions.QiskitBackendNotFoundError: 'No backend matches the criteria.'\n"

## Output
Output bagi Fungsi Circuit adalah [PrimitiveResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult), yang mengandungi dua medan:

- Satu atau lebih objek [PubResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult). Ini boleh diindeks secara langsung dari `PrimitiveResult`.

- Metadata peringkat kerja.

Setiap `PubResult` mengandungi medan `data` dan `metadata`.

- Medan `data` mengandungi sekurang-kurangnya satu tatasusunan nilai jangkaan (`PubResult.data.evs`) dan satu tatasusunan ralat piawai (`PubResult.data.stds`). Ia juga boleh mengandungi lebih banyak data, bergantung pada pilihan yang digunakan.

- Medan `metadata` mengandungi metadata peringkat PUB (`PubResult.metadata`).

Petikan kod berikut menerangkan format `PrimitiveResult` (dan `PubResult` yang berkaitan).